### hetero graphsage vs r-gcn, 5 fold cv
reads: ../prep/, ../results/cv_folds.csv (r-gcn from 05_cv)  
writes: graphsage/results/  
same folds, splits, negatives, loss and metrics as 05_cv, only the graph layer changes

#### setup

In [ ]:
import os, json, time, numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt

PREP = 'C:/Users/Nimishavarma/OneDrive/Dokumenty/hti-project/prep'
RGCN = 'C:/Users/Nimishavarma/OneDrive/Dokumenty/hti-project/results/cv_folds.csv'
OUT  = 'C:/Users/Nimishavarma/OneDrive/Dokumenty/hti-project/graphsage/results'
os.makedirs(OUT, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

K = 5
BATCH, EPOCHS, PATIENCE = 1024, 300, 20
AGGRS = ['mean', 'pool']
print(device)

#### load data (same as 05_cv)

In [ ]:
meta = json.load(open(f'{PREP}/meta.json'))
n_h, n_c, n_t = meta['n_h'], meta['n_c'], meta['n_t']
N = n_h + n_c + n_t

C_feat = torch.from_numpy(np.load(f'{PREP}/C_feat.npy')).to(device)
T_feat = torch.from_numpy(np.load(f'{PREP}/T_feat.npy')).to(device)
hc = np.load(f'{PREP}/hc_edges.npy')

ct_all = np.concatenate([np.load(f'{PREP}/{x}.npy') for x in ['ct_train','ct_val_pos','ct_test_pos']], axis=1)
pos_key = ct_all[0] * n_t + ct_all[1]
neg_key = np.setdiff1d(np.arange(n_c * n_t), pos_key)

known = torch.zeros(n_c, n_t, dtype=torch.bool, device=device)
known[torch.from_numpy(ct_all[0]).long(), torch.from_numpy(ct_all[1]).long()] = True
print(ct_all.shape, len(neg_key))

#### herb-target setup

In [ ]:
H_df = pd.DataFrame(hc.T, columns=['h','c'])

def ht_keys(ct):
    j = H_df.merge(pd.DataFrame(ct.T, columns=['c','t']), on='c')
    return np.unique(j['h'].values * n_t + j['t'].values)

ht_all = ht_keys(ct_all)
ht_known = np.zeros(n_h * n_t, dtype=bool)
ht_known[ht_all] = True
ht_known = ht_known.reshape(n_h, n_t)
ht_neg = np.setdiff1d(np.arange(n_h * n_t), ht_all)
herb_cmps = [torch.from_numpy(np.unique(hc[1][hc[0] == h])).long().to(device) for h in range(n_h)]

#### hetero graphsage layer
per relation: own vector through one weight, neighbours combined (mean or pool) through another, results summed over relations that end at the same node type

In [ ]:
# node index ranges, same layout as before: herbs, compounds, targets
RANGE = {'H': (0, n_h), 'C': (n_h, n_h + n_c), 'T': (n_h + n_c, N)}
# relation order matches make_esets: H->C, C->H, C->T, T->C
RELS = [('H', 'C'), ('C', 'H'), ('C', 'T'), ('T', 'C')]

class SAGELayer(nn.Module):
    def __init__(self, d, aggr):
        super().__init__()
        self.aggr = aggr
        self.root = nn.ModuleList([nn.Linear(d, d) for _ in RELS])
        self.nei  = nn.ModuleList([nn.Linear(d, d, bias=False) for _ in RELS])
        if aggr == 'pool':
            self.pool = nn.ModuleList([nn.Linear(d, d) for _ in RELS])

    def forward(self, x, es):
        parts = {'H': 0, 'C': 0, 'T': 0}
        for r, ((src, dst), (_, dt)) in enumerate(zip(es, RELS)):
            if self.aggr == 'mean':
                agg = torch.zeros_like(x).index_add_(0, dst, x[src])
                deg = torch.zeros(x.size(0), device=x.device).index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))
                agg = agg / deg.clamp(min=1).unsqueeze(1)
            else:
                m = F.relu(self.pool[r](x[src]))
                agg = torch.zeros_like(x).scatter_reduce(0, dst[:, None].expand(-1, x.size(1)), m, reduce='amax', include_self=False)
            lo, hi = RANGE[dt]
            parts[dt] = parts[dt] + self.root[r](x[lo:hi]) + self.nei[r](agg[lo:hi])
        return torch.cat([parts['H'], parts['C'], parts['T']], dim=0)

class Model(nn.Module):
    def __init__(self, aggr, d=128, dropout=0.2):
        super().__init__()
        self.herb_emb = nn.Embedding(n_h, d)
        self.proj_c = nn.Linear(C_feat.shape[1], d)
        self.proj_t = nn.Linear(T_feat.shape[1], d)
        self.l1 = SAGELayer(d, aggr)
        self.l2 = SAGELayer(d, aggr)
        self.drop = nn.Dropout(dropout)
        self.rel = nn.Parameter(torch.randn(d) * 0.01)
    def encode(self, es):
        x = torch.cat([self.herb_emb.weight, F.relu(self.proj_c(C_feat)), F.relu(self.proj_t(T_feat))], dim=0)
        x = self.drop(F.relu(self.l1(x, es)))
        return self.l2(x, es)
    def score_ct(self, Z, c, t):
        return (Z[n_h + c] * self.rel * Z[n_h + n_c + t]).sum(-1)
    def scores_all(self, Z):
        return (Z[n_h:n_h+n_c] * self.rel) @ Z[n_h+n_c:].t()

#### split, sampling, metric helpers (same as 05_cv)

In [ ]:
def split_eh(rng):
    p = rng.permutation(ct_all.shape[1])
    a, b = int(0.70 * len(p)), int(0.85 * len(p))
    return ct_all[:, p[:a]], ct_all[:, p[a:b]], ct_all[:, p[b:]]

def split_cd(rng):
    cs = rng.permutation(np.unique(ct_all[0]))
    a, b = int(0.70 * len(cs)), int(0.85 * len(cs))
    parts = [cs[:a], cs[a:b], cs[b:]]
    return [ct_all[:, np.isin(ct_all[0], x)] for x in parts], parts

def sample_neg_pairs(rng, n, cmps=None):
    pool = neg_key if cmps is None else neg_key[np.isin(neg_key // n_t, cmps)]
    k = rng.choice(pool, size=min(n, len(pool)), replace=False)
    return np.stack([k // n_t, k % n_t])

def tt(a):
    return torch.from_numpy(np.asarray(a)).long().to(device)

def make_esets(ct_tr):
    h, c = tt(hc[0]), tt(hc[1] + n_h)
    cc, t = tt(ct_tr[0] + n_h), tt(ct_tr[1] + n_h + n_c)
    return [(h, c), (c, h), (cc, t), (t, cc)]

def pr_auc(sp, sn):
    y = np.r_[np.ones(len(sp)), np.zeros(len(sn))]
    return float(average_precision_score(y, np.r_[sp, sn]))

def rank_metrics(r):
    r = np.asarray(r, dtype=np.float64)
    return {'Hits@10': float((r <= 10).mean()), 'Hits@50': float((r <= 50).mean()), 'MRR': float((1 / r).mean())}

#### train one model (bpr loss, early stop on val pr-auc, same as 05_cv)

In [ ]:
def train(aggr, ct_tr, vp, vn, seed):
    torch.manual_seed(seed)
    es = make_esets(ct_tr)
    m = Model(aggr).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-5)
    tr, vp_t, vn_t = tt(ct_tr), tt(vp), tt(vn)

    best, wait, best_ep, state = -1, 0, 0, None
    for ep in range(1, EPOCHS + 1):
        m.train()
        perm = torch.randperm(tr.shape[1], device=device)
        for i in range(0, len(perm), BATCH):
            b = perm[i:i+BATCH]
            Z = m.encode(es)
            t_neg = torch.randint(0, n_t, (len(b),), device=device)
            loss = -F.logsigmoid(m.score_ct(Z, tr[0, b], tr[1, b]) - m.score_ct(Z, tr[0, b], t_neg)).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()

        if ep % 2 == 0:
            m.eval()
            with torch.no_grad():
                Z = m.encode(es)
                pr = pr_auc(m.score_ct(Z, vp_t[0], vp_t[1]).cpu().numpy(), m.score_ct(Z, vn_t[0], vn_t[1]).cpu().numpy())
            if pr > best:
                best, wait, best_ep = pr, 0, ep
                state = {k: v.clone() for k, v in m.state_dict().items()}
            else:
                wait += 1
                if wait >= PATIENCE:
                    break

    m.load_state_dict(state)
    m.eval()
    with torch.no_grad():
        S = m.scores_all(m.encode(es))
    return S, best, best_ep

#### eval on a full score matrix (filtered ranking for ct, test-only pairs for ht)

In [ ]:
@torch.no_grad()
def eval_ct(S, te, tn):
    c, t = tt(te[0]), tt(te[1])
    s_true = S[c, t]
    filt = 1 + ((S[c] > s_true[:, None]) & ~known[c]).sum(1)
    sn = S[tt(tn[0]), tt(tn[1])]
    out = {'PR-AUC': pr_auc(s_true.cpu().numpy(), sn.cpu().numpy())}
    out.update(rank_metrics(filt.cpu().numpy()))
    return out

@torch.no_grad()
def eval_ht(S, ct_tr, ct_va, ct_te):
    HS = torch.stack([S[c].max(0).values for c in herb_cmps]).cpu().numpy()
    seen = np.union1d(ht_keys(ct_tr), ht_keys(ct_va))
    test = np.setdiff1d(ht_keys(ct_te), seen)
    flat = HS.ravel()
    out = {'PR-AUC': pr_auc(flat[test], flat[ht_neg])}
    h, t = test // n_t, test % n_t
    filt = 1 + ((HS[h] > HS[h, t][:, None]) & ~ht_known[h]).sum(1)
    out.update(rank_metrics(filt))
    return out

def popularity(ct_tr):
    # score = how many training compounds hit the target, tiny noise to break ties
    deg = torch.bincount(tt(ct_tr[1]), minlength=n_t).float()
    noise = 1e-6 * torch.rand(n_c, n_t, generator=torch.Generator().manual_seed(0)).to(device)
    return deg.unsqueeze(0).expand(n_c, -1) + noise

def row(model, split, k, S, tr, va, te, tn, v=np.nan, ep=np.nan):
    r = {'fold': k, 'split': split, 'model': model, 'val PR-AUC': v, 'best_ep': ep}
    r.update({f'CT {a}': b for a, b in eval_ct(S, te, tn).items()})
    r.update({f'HT {a}': b for a, b in eval_ht(S, tr, va, te).items()})
    return r

#### cv loop: 5 folds x 2 splits x (sage-mean, sage-pool, popularity)
same rng sequence as 05_cv, so the folds are identical

In [ ]:
rows = []
t0 = time.time()
for k in range(K):
    rng = np.random.default_rng(100 + k)

    tr, va, te = split_eh(rng)
    vn = sample_neg_pairs(rng, 10 * va.shape[1])
    tn = sample_neg_pairs(rng, 50 * te.shape[1])
    rows.append(row('popularity', 'edge_hiding', k, popularity(tr), tr, va, te, tn))
    for ag in AGGRS:
        S, v, ep = train(ag, tr, va, vn, seed=k)
        rows.append(row(f'sage-{ag}', 'edge_hiding', k, S, tr, va, te, tn, v, ep))
        print(f"fold {k} EH sage-{ag:4s}  ct h@10 {rows[-1]['CT Hits@10']:.4f}  ht h@10 {rows[-1]['HT Hits@10']:.4f}  ep {ep}  {time.time()-t0:.0f}s")

    (tr, va, te), parts = split_cd(rng)
    vn = sample_neg_pairs(rng, 10 * va.shape[1], parts[1])
    tn = sample_neg_pairs(rng, 50 * te.shape[1], parts[2])
    rows.append(row('popularity', 'compound_disjoint', k, popularity(tr), tr, va, te, tn))
    for ag in AGGRS:
        S, v, ep = train(ag, tr, va, vn, seed=k)
        rows.append(row(f'sage-{ag}', 'compound_disjoint', k, S, tr, va, te, tn, v, ep))
        print(f"fold {k} CD sage-{ag:4s}  ct h@10 {rows[-1]['CT Hits@10']:.4f}  ht h@10 {rows[-1]['HT Hits@10']:.4f}  ep {ep}  {time.time()-t0:.0f}s")

sage = pd.DataFrame(rows)
sage.to_csv(f'{OUT}/sage_cv_folds.csv', index=False)

#### compare with r-gcn from 05_cv

In [ ]:
cols = ['CT PR-AUC','CT Hits@10','CT Hits@50','CT MRR','HT PR-AUC','HT Hits@10','HT Hits@50','HT MRR']
rgcn = pd.read_csv(RGCN).assign(model='r-gcn')
allm = pd.concat([rgcn[['fold','split','model'] + cols], sage[['fold','split','model'] + cols]])

mu = allm.groupby(['split','model'])[cols].mean()
sd = allm.groupby(['split','model'])[cols].std()
table = mu.round(4).astype(str) + ' +/- ' + sd.round(4).astype(str)
order = ['r-gcn', 'sage-mean', 'sage-pool', 'popularity']
table = table.reindex([(s, m) for s in ['edge_hiding', 'compound_disjoint'] for m in order])
table.to_csv(f'{OUT}/compare.csv')
table.T

#### plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
x = np.arange(len(cols))
w = 0.2
for ax, sp in zip(axes, ['edge_hiding', 'compound_disjoint']):
    for i, mdl in enumerate(order):
        ax.bar(x + (i - 1.5) * w, mu.loc[(sp, mdl)], w, yerr=sd.loc[(sp, mdl)], capsize=2, label=mdl)
    ax.set_xticks(x)
    ax.set_xticklabels(cols, rotation=35)
    ax.set_title(sp)
axes[0].legend()
fig.tight_layout()
fig.savefig(f'{OUT}/compare.png', dpi=120)
plt.show()